# Тема 6. Подсистема памяти агента и технология RAG

**Модуль II · Пример 3 из 4**

Без памяти каждый вызов языковой модели начинается «с нуля». Память превращает stateless-модель в агента. Разберём:

**Память**
- **Кратковременная** (рабочая) — контекст текущей сессии.
- **Долговременная** — знания между сессиями; в т. ч. **память о сущностях** (entity memory): факты о пользователе и объектах.
- **Контекстная** — то, что фактически попадает в контекстное окно на данном шаге.

**RAG (Retrieval-Augmented Generation)** — связующее звено между долговременной памятью и контекстом: агент **ищет** релевантные знания во внешнем корпусе и подмешивает их в запрос. Разберём офлайн-индекс (эмбеддинги + BM25), онлайн-поиск, **двухстадийный поиск с переранжированием** и преобразования запроса (**переписывание** и **HyDE**).

> Корпус знаний — реальные абзацы из статей Википедии (датасет **SQuAD**). Память — про клиента магазина «ШопБот». Эмбеддинги вычисляются моделью `text-embedding-3-small` через OpenAI-совместимый API.

### Библиотеки
- **numpy** — векторные операции, косинусная близость.
- **rank_bm25** — лексический индекс BM25.
- **openai** — и языковая модель, и эмбеддинги.

## Часть A. Память агента

### Шаг 0. Конфигурация

In [1]:
import os, json, time
import numpy as np
from openai import OpenAI

API_KEY = os.environ.get("LLM_API_KEY", "YOUR_API_KEY")
client = OpenAI(api_key=API_KEY, base_url=os.environ.get("LLM_BASE_URL", "YOUR_API_BASE_URL"))
MODEL = "qwen/qwen3.7-max"
EMBED_MODEL = "text-embedding-3-small"
print("Готово.")

Готово.


### Шаг 1. Кратковременная (рабочая) память

Это история текущей сессии. Проблема: контекстное окно ограничено. Простейшая стратегия управления — **скользящее окно (token buffer)**: храним только последние N сообщений. Реализуем через `collections.deque` с ограничением длины: старые сообщения вытесняются автоматически.

In [2]:
from collections import deque

class ShortTermMemory:
    """Рабочая память сессии со скользящим окном последних N сообщений."""
    def __init__(self, window: int = 6):
        self.buffer = deque(maxlen=window)   # maxlen сам вытесняет старое
    def add(self, role: str, content: str):
        self.buffer.append({"role": role, "content": content})
    def messages(self) -> list[dict]:
        return list(self.buffer)

stm = ShortTermMemory(window=4)
for i in range(6):
    stm.add("user", f"сообщение {i}")
print("В окне осталось (последние 4):", [m["content"] for m in stm.messages()])

В окне осталось (последние 4): ['сообщение 2', 'сообщение 3', 'сообщение 4', 'сообщение 5']


### Шаг 2. Долговременная память и память о сущностях (entity memory)

Долговременная память живёт **между сессиями** во внешнем хранилище. Частный случай — **entity memory**: структурированные факты о пользователе (имя, предпочтения) и объектах. Реализуем простое персистентное хранилище на JSON-файле: профиль сохраняется и доступен в следующей сессии.

In [3]:
from pathlib import Path

class LongTermMemory:
    """Персистентная долговременная память (профиль + факты о сущностях) в JSON-файле."""
    def __init__(self, path="memory_store.json"):
        self.path = Path(path)
        self.data = json.loads(self.path.read_text()) if self.path.exists() else {"profile": {}, "entities": {}}
    def set_profile(self, **kw):
        self.data["profile"].update(kw); self._save()
    def remember_entity(self, name: str, **facts):
        self.data["entities"].setdefault(name, {}).update(facts); self._save()
    def _save(self):
        self.path.write_text(json.dumps(self.data, ensure_ascii=False, indent=2))
    def as_context(self) -> str:
        return json.dumps(self.data, ensure_ascii=False)

ltm = LongTermMemory()
ltm.set_profile(name="Иван", city="Санкт-Петербург", loyalty="gold")
ltm.remember_entity("заказ ORD-1001", товар="T-Light Holder", количество=2, статус="отправлен")
print("Сохранённая долговременная память:\n", ltm.as_context())

Сохранённая долговременная память:
 {"profile": {"name": "Иван", "city": "Санкт-Петербург", "loyalty": "gold"}, "entities": {"заказ ORD-1001": {"товар": "T-Light Holder", "количество": 2, "статус": "отправлен"}}}


### Шаг 3. Контекстная память = что реально уходит в модель

Контекстная память собирается в рантайме из источников: системная инструкция + долговременная память + окно диалога. Соберём ассистента, который **персонализирует** ответ, зная профиль из долговременной памяти, даже в новой сессии.

In [4]:
def assistant_reply(user_message: str, stm: ShortTermMemory, ltm: LongTermMemory) -> str:
    system = ("Ты — ассистент магазина ШопБот. Учитывай профиль клиента и историю. "
              "Профиль и известные сущности (JSON): " + ltm.as_context())
    messages = [{"role": "system", "content": system}] + stm.messages() + \
               [{"role": "user", "content": user_message}]
    resp = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=300)
    answer = resp.choices[0].message.content
    stm.add("user", user_message); stm.add("assistant", answer)   # пополняем рабочую память
    return answer

# НОВАЯ сессия: рабочая память пуста, но долговременная — из файла
session = ShortTermMemory(window=6)
print(assistant_reply("Привет! Напомни, какой у меня статус лояльности и что с последним заказом?",
                      session, LongTermMemory()))

Привет, Иван! 👋

Ваш статус лояльности — **Gold** (золотой). Спасибо, что вы с нами! 🌟

Что касается вашего последнего заказа **ORD-1001** (2 шт. T-Light Holder), он уже **отправлен** и находится в пути к вам в Санкт-Петербург. 📦

Подсказать трек-номер для отслеживания посылки или помочь с чем-нибудь ещё?


## Часть B. RAG — поиск знаний во внешнем корпусе

### Шаг 4. Корпус знаний (offline-подготовка)

Возьмём реальный корпус — абзацы из статей Википедии (SQuAD) по нескольким темам. В офлайне из корпуса строят **поисковые индексы**. Мы построим два: **векторный** (семантический) и **BM25** (лексический).

In [5]:
sq = json.load(open("data/squad_dev.json"))
TOPICS = {"Nikola_Tesla", "Oxygen", "Steam_engine", "Apollo_program", "Normans"}
CORPUS = []
for art in sq["data"]:
    if art["title"] in TOPICS:
        for p in art["paragraphs"]:
            CORPUS.append(p["context"])
# уберём дубли, укоротим до разумного размера
CORPUS = list(dict.fromkeys(CORPUS))
print("Тем:", len(TOPICS), "| абзацев в корпусе:", len(CORPUS))
print("Пример абзаца:\n", CORPUS[0][:200], "...")

Тем: 5 | абзацев в корпусе: 283
Пример абзаца:
 The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse (" ...


### Шаг 5. Векторный индекс (эмбеддинги + косинусная близость)

**Эмбеддинг** — числовой вектор, кодирующий смысл текста; близкие по смыслу тексты дают близкие векторы. Считаем эмбеддинги абзацев (батчами через API) и **кэшируем** в `.npy`, чтобы не пересчитывать. Поиск — по **косинусной близости** запроса ко всем абзацам.

In [6]:
def embed(texts: list[str], batch: int = 96) -> np.ndarray:
    """Эмбеддинги списка текстов, батчами. Возвращает матрицу (n, dim)."""
    vecs = []
    for i in range(0, len(texts), batch):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i+batch])
        vecs.extend([d.embedding for d in r.data])
    return np.array(vecs, dtype=np.float32)

CACHE = Path("data/rag_embeddings.npy")
if CACHE.exists():
    DOC_VECS = np.load(CACHE)
    print("Эмбеддинги загружены из кэша:", DOC_VECS.shape)
else:
    t = time.time()
    DOC_VECS = embed(CORPUS)
    np.save(CACHE, DOC_VECS)
    print(f"Посчитано {DOC_VECS.shape[0]} эмбеддингов за {time.time()-t:.1f} c, размерность {DOC_VECS.shape[1]}")

# нормируем векторы -> косинусная близость сводится к скалярному произведению.
# Для скалярных произведений ниже используем np.einsum (а не оператор @):
# на некоторых сборках NumPy 2.x оператор @ выдаёт ложные RuntimeWarning.
DOC_NORM = (DOC_VECS / np.linalg.norm(DOC_VECS, axis=1, keepdims=True)).astype(np.float64)

def normalize(vec: np.ndarray) -> np.ndarray:
    return (vec / np.linalg.norm(vec)).astype(np.float64)

def vector_search(query: str, k: int = 3):
    q = normalize(embed([query])[0])
    scores = np.einsum("ij,j->i", DOC_NORM, q)   # косинусная близость ко всем абзацам
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), CORPUS[i]) for i in top]

for idx, score, text in vector_search("Where was Nikola Tesla born?"):
    print(f"[{score:.3f}] {text[:110]}...")

Эмбеддинги загружены из кэша: (283, 1536)


[0.735] Tesla was born on 10 July [O.S. 28 June] 1856 into a Serb family in the village of Smiljan, Austrian Empire (m...
[0.697] Nikola Tesla (Serbian Cyrillic: Никола Тесла; 10 July 1856 – 7 January 1943) was a Serbian American inventor, ...
[0.649] Tesla was the fourth of five children. He had an older brother named Dane and three sisters, Milka, Angelina a...


### Шаг 6. Лексический индекс BM25

**BM25** ранжирует документы по совпадению **ключевых слов** (частота термина с поправкой на длину). Он хорош там, где важно точное слово (имя, код), но «не понимает» синонимы. Библиотека **`rank_bm25`** строит индекс по токенизированным документам.

In [7]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

BM25 = BM25Okapi([tokenize(doc) for doc in CORPUS])

def bm25_search(query: str, k: int = 3):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), CORPUS[i]) for i in top]

for idx, score, text in bm25_search("liquid oxygen boiling point", k=3):
    print(f"[{score:.2f}] {text[:110]}...")

[10.86] In 1891 Scottish chemist James Dewar was able to produce enough liquid oxygen to study. The first commercially...
[8.86] Oxygen storage methods include high pressure oxygen tanks, cryogenics and chemical compounds. For reasons of e...
[7.90] Oxygen condenses at 90.20 K (−182.95 °C, −297.31 °F), and freezes at 54.36 K (−218.79 °C, −361.82 °F). Both li...


### Шаг 7. Двухстадийный поиск (recall → re-ranking)

Идея: сначала **широкий отбор** дешёвым методом (высокий recall), затем **точное переранжирование** более качественным (высокая precision). Здесь: Stage 1 — BM25 даёт top-50 кандидатов; Stage 2 — переранжируем их по **семантической** близости эмбеддингов. Так объединяем сильные стороны лексического и векторного поиска.

In [8]:
def two_stage_search(query: str, recall_k: int = 50, final_k: int = 3):
    # Stage 1: быстрый широкий отбор кандидатов лексическим BM25
    scores = BM25.get_scores(tokenize(query))
    candidates = np.argsort(-scores)[:recall_k]
    # Stage 2: точное переранжирование кандидатов по эмбеддингам
    q = normalize(embed([query])[0])
    cand_vecs = np.ascontiguousarray(DOC_NORM[candidates])
    rerank = np.einsum("ij,j->i", cand_vecs, q)
    order = candidates[np.argsort(-rerank)][:final_k]
    return [(int(i), CORPUS[i]) for i in order]

print("Двухстадийный поиск по запросу 'how did the Apollo program land humans on the Moon':")
for idx, text in two_stage_search("how did the Apollo program land humans on the Moon"):
    print(" -", text[:120], "...")

Двухстадийный поиск по запросу 'how did the Apollo program land humans on the Moon':


 - The Apollo program, also known as Project Apollo, was the third United States human spaceflight program carried out by t ...
 - Apollo set several major human spaceflight milestones. It stands alone in sending manned missions beyond low Earth orbit ...
 - The Apollo program succeeded in achieving its goal of manned lunar landing, despite the major setback of a 1967 Apollo 1 ...


### Шаг 8. Преобразование запроса №1 — переписывание (query rewriting)

Проблема: в диалоге запросы **контекстно-зависимы** («А где он родился?»). Векторный поиск по такому запросу промахивается — в нём нет ключевой сущности. Решение: языковой моделью **переписать** запрос в самодостаточную (standalone) форму с учётом истории.

In [9]:
def rewrite_query(history: list[dict], follow_up: str) -> str:
    prompt = ("Перепиши последний вопрос пользователя в самодостаточную форму "
              "(подставь сущности из истории), верни ТОЛЬКО переписанный вопрос на английском.\n\n"
              "История:\n" + "\n".join(f"{m['role']}: {m['content']}" for m in history) +
              f"\n\nВопрос: {follow_up}")
    r = client.chat.completions.create(model=MODEL,
            messages=[{"role": "user", "content": prompt}], max_tokens=60)
    return r.choices[0].message.content.strip()

history = [{"role": "user", "content": "Tell me about Nikola Tesla"},
           {"role": "assistant", "content": "Nikola Tesla was an inventor and electrical engineer."}]
follow_up = "And where was he born?"

print("Наивный поиск по '%s':" % follow_up)
for _, s, text in vector_search(follow_up, k=1):
    print(f"  [{s:.3f}] {text[:90]}...")

standalone = rewrite_query(history, follow_up)
print("\nПереписанный запрос:", standalone)
print("Поиск по переписанному запросу:")
for _, s, text in vector_search(standalone, k=1):
    print(f"  [{s:.3f}] {text[:90]}...")

Наивный поиск по 'And where was he born?':


  [0.323] Tesla was born on 10 July [O.S. 28 June] 1856 into a Serb family in the village of Smiljan...



Переписанный запрос: Where was Nikola Tesla born?
Поиск по переписанному запросу:


  [0.735] Tesla was born on 10 July [O.S. 28 June] 1856 into a Serb family in the village of Smiljan...


### Шаг 9. Преобразование запроса №2 — HyDE

**HyDE (Hypothetical Document Embeddings)**: короткий разговорный вопрос семантически «далёк» от формального абзаца-ответа. Приём — попросить модель сгенерировать **гипотетический ответ** (пусть даже с неточностями) и искать по **его** эмбеддингу: гипотетический ответ ближе по форме к реальному документу, чем исходный вопрос.

In [10]:
def hyde_search(question: str, k: int = 3):
    # 1) модель генерирует гипотетический ответ-абзац
    r = client.chat.completions.create(model=MODEL, max_tokens=120,
        messages=[{"role": "user", "content":
            f"Напиши краткий энциклопедический абзац-ответ на вопрос (по-английски): {question}"}])
    hypothetical = r.choices[0].message.content.strip()
    # 2) ищем по эмбеддингу гипотетического ответа, а не самого вопроса
    q = normalize(embed([hypothetical])[0])
    scores = np.einsum("ij,j->i", DOC_NORM, q)
    top = np.argsort(-scores)[:k]
    return hypothetical, [(int(i), float(scores[i]), CORPUS[i]) for i in top]

q = "What makes the steam engine important for industry?"
hyp, hits = hyde_search(q)
print("Гипотетический ответ (для поиска):\n ", hyp[:200], "...\n")
print("Найденные абзацы:")
for _, s, text in hits[:2]:
    print(f"  [{s:.3f}] {text[:110]}...")

Гипотетический ответ (для поиска):
  The steam engine was the foundational catalyst of the First Industrial Revolution, fundamentally transforming global industry by providing a reliable, continuous, and location-independent source of me ...

Найденные абзацы:
  [0.742] Steam engines can be said to have been the moving force behind the Industrial Revolution and saw widespread co...
  [0.697] In 1781 James Watt patented a steam engine that produced continuous rotary motion. Watt's ten-horsepower engin...


### Шаг 10. Полный RAG-ответ

Соберём конвейер: поиск (двухстадийный) → сборка контекста → генерация ответа **строго по найденным абзацам**. Это и есть Retrieval-Augmented Generation: модель отвечает, опираясь на извлечённые знания, а не на «память весов».

In [11]:
def rag_answer(question: str, k: int = 3) -> str:
    hits = two_stage_search(question, final_k=k)
    context = "\n\n".join(f"[{n+1}] {text}" for n, (_, text) in enumerate(hits))
    prompt = (f"Ответь на вопрос, используя ТОЛЬКО факты из контекста. "
              f"Если ответа в контексте нет — так и скажи.\n\nКонтекст:\n{context}\n\nВопрос: {question}")
    r = client.chat.completions.create(model=MODEL, max_tokens=250,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print(rag_answer("Where and when was Nikola Tesla born?"))

Никола Тесла родился 10 июля [по старому стилю 28 июня] 1856 года в деревне Смилян, Австрийская империя (современная Хорватия).


### Шаг 11. RAG vs дообучение и другие стратегии контекста

**RAG или дообучение?** Практическое правило: **сначала RAG** (быстро настроить, знания всегда актуальны, ответ обоснован источником). Дообучение (Continued Pretraining / SFT / LoRA) оправдано, когда домен стабилен, нужна глубокая экспертиза и есть данные + вычислительные ресурсы.

**Стратегии управления контекстом**, когда история не влезает в окно:
- **Скользящее окно** (реализовано в Шаге 1) — просто, но теряет старое;
- **Суммаризация** — периодически сжимать старый диалог моделью (ниже);
- **Векторная память** — извлекать только релевантные прошлые сообщения (тот же механизм RAG, но по истории);
- **Модели с длинным контекстом** — ничего не терять ценой стоимости и задержки.

In [12]:
def summarize_dialog(messages: list[dict]) -> str:
    text = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    r = client.chat.completions.create(model=MODEL, max_tokens=120,
        messages=[{"role": "user", "content": "Сожми диалог в 2-3 факта для памяти:\n" + text}])
    return r.choices[0].message.content

demo = [{"role": "user", "content": "Меня зовут Иван, я из Петербурга"},
        {"role": "assistant", "content": "Приятно познакомиться, Иван!"},
        {"role": "user", "content": "Люблю товары для дома и чай"}]
print("Саммари для долговременной памяти:\n", summarize_dialog(demo))

Саммари для долговременной памяти:
 1. Иван из Петербурга.
2. Любит товары для дома и чай.


## Итоги

- Память: **кратковременная** (сессия, скользящее окно) → **долговременная** (персистентная, entity memory) → **контекстная** (что реально уходит в модель).
- **RAG** связывает долговременную память и контекст: офлайн — индексы (**эмбеддинги** + **BM25**), онлайн — поиск → (переранжирование) → генерация по найденному.
- **Двухстадийный поиск**: дешёвый recall (BM25) + точный re-ranking (эмбеддинги).
- Преобразования запроса: **переписывание** спасает контекстно-зависимые запросы, **HyDE** сближает вопрос с формой документа.
- Выбор **RAG vs дообучение** и стратегии контекста — это компромисс «качество — задержка — стоимость».

**Дальше:** Тема 7 — защитный контур: как защитить агента от инъекций, утечек и галлюцинаций.